# CELL 1 — Import & Mount Google Drive

In [ ]:
import os
import json
import random
import requests
import time
from google.colab import drive
from google.colab import userdata

drive.mount('/content/drive', force_remount=True)
print("Google Drive berhasil di-mount dengan akses penuh!")

Mounted at /content/drive
Google Drive berhasil di-mount dengan akses penuh!


# CELL 2 — Parameter, Konfigurasi API & Direktori


In [ ]:
# 1. Parameter Utama
PILIHAN_MODEL  = "gemini"
MATA_PELAJARAN = "IPS" # sesuaikan

# 2. Opsi Pengacakan Skenario
OPSI_KOGNITIF = ["low", "middle", "high"]
OPSI_EMOSI    = ["antusias", "bingung", "bosan", "frustrasi", "tidak_terdeteksi"]
OPSI_INSIDEN  = ["0x", "1x", "2x"]

# 3. Konfigurasi Direktori
ROOT_DIR = "/content/drive/MyDrive/sft_v4/" #sesuaikan nama folder prompt
DIR_SUMBER_DATA = os.path.join(ROOT_DIR, f"output_{MATA_PELAJARAN.lower()}_mpe/") #sesuaikan nama folder sumber

# 4. Konfigurasi API
if PILIHAN_MODEL == "gemini":
    MODEL_NAME = "gemini-2.5-flash-lite"
    API_KEY    = userdata.get('TOKEN_API')
    URL        = "https://openrouter.ai/api/v1/chat/completions"
    print(f"Model aktif: {MODEL_NAME}")

Model aktif: gemini-2.5-flash-lite


# CELL 3 — Scan File Sumber



In [ ]:
import os
import glob
import random
import json
import requests
import time

if 'hasil_dataset' not in globals():
    hasil_dataset = []

# Fungsi pencari file di dalam output_ips_mpe
def kumpulkan_file_sumber(base_dir):
    kumpulan_file = []
    # Cari ke semua subfolder
    search_pattern = os.path.join(base_dir, "**", "*.json")
    semua_file = glob.glob(search_pattern, recursive=True)

    for file_path in semua_file:
        if "/essay/" in file_path:
            kumpulan_file.append(file_path)

    return kumpulan_file

daftar_file_sumber = kumpulkan_file_sumber(DIR_SUMBER_DATA)
print(f"Berhasil menemukan {len(daftar_file_sumber)} file essay.")

Berhasil menemukan 810 file essay.


# CELL 4 — Perakit Prompt

In [ ]:
def baca_md(path_file): #baca isi file .md
    try:
        with open(path_file, "r", encoding="utf-8") as f:
            return f.read().strip()
    except FileNotFoundError:
        return ""

def get_prompts_dinamis(kognitif, emosi, oot, bad, teks_konteks_jadi, prefix_siswa):
    with open(PATH_BASE_SYS, "r", encoding="utf-8") as f:
        sys_prompt = f.read()
    with open(PATH_BASE_USR, "r", encoding="utf-8") as f:
        usr_prompt = f.read()

    # RAKIT KOMPONEN
    # 1. membuka file base_system dan base_user
    # 2. membaca file-file komponen
    teks_treat_kognitif = baca_md(os.path.join(DIR_KOMPONEN_SYS, f"treatment_kognitif/{kognitif}.md"))
    teks_treat_emosi    = baca_md(os.path.join(DIR_KOMPONEN_SYS, f"treatment_emosi/{emosi}.md"))
    teks_aturan_oot = baca_md(os.path.join(DIR_KOMPONEN_SYS, f"aturan_oot_bad/oot_{oot}.md")) if oot != "0x" else ""
    teks_aturan_bad = baca_md(os.path.join(DIR_KOMPONEN_SYS, f"aturan_oot_bad/bad_{bad}.md")) if bad != "0x" else ""

    larangan_emosi = baca_md(os.path.join(DIR_KOMPONEN_SYS, f"larangan_tambahan/{emosi}.md"))
    larangan_oot   = baca_md(os.path.join(DIR_KOMPONEN_SYS, f"larangan_tambahan/oot.md")) if oot != "0x" else ""
    teks_larangan  = f"{larangan_emosi}\n{larangan_oot}".strip()

    if bad != "0x": teks_struktur = baca_md(os.path.join(DIR_KOMPONEN_SYS, "struktur_respons/bad.md"))
    elif oot != "0x": teks_struktur = baca_md(os.path.join(DIR_KOMPONEN_SYS, "struktur_respons/oot.md"))
    else: teks_struktur = baca_md(os.path.join(DIR_KOMPONEN_SYS, f"struktur_respons/{emosi}.md"))

    teks_karakter_kognitif = baca_md(os.path.join(DIR_KOMPONEN_USR, f"karakter_siswa/kognitif/{kognitif}.md"))
    teks_karakter_emosi    = baca_md(os.path.join(DIR_KOMPONEN_USR, f"karakter_siswa/emosi/{emosi}.md"))
    teks_karakter_oot = baca_md(os.path.join(DIR_KOMPONEN_USR, f"karakter_siswa/konteks/oot_{oot}.md")) if oot != "0x" else ""
    teks_karakter_bad = baca_md(os.path.join(DIR_KOMPONEN_USR, f"karakter_siswa/konteks/bad_{bad}.md")) if bad != "0x" else ""

    # 3. mencari tulisan seperti [LABEL_EMOSI], menggantinya dengan teks komponen asli menggunakan perintah .replace()
    # SUNTIK KE SYSTEM
    sys_prompt = sys_prompt.replace("[LABEL_KOGNITIF]", kognitif.upper())
    sys_prompt = sys_prompt.replace("[LABEL_EMOSI]", emosi.upper())
    sys_prompt = sys_prompt.replace("[JUMLAH_OOT]", oot)
    sys_prompt = sys_prompt.replace("[JUMLAH_BAD]", bad)
    sys_prompt = sys_prompt.replace("[ISI_KONTEKS]", teks_konteks_jadi)
    sys_prompt = sys_prompt.replace("[TEKS_TREATMENT_KOGNITIF]", teks_treat_kognitif)
    sys_prompt = sys_prompt.replace("[TEKS_TREATMENT_EMOSI]", teks_treat_emosi)
    sys_prompt = sys_prompt.replace("[TEKS_ATURAN_OOT]", teks_aturan_oot)
    sys_prompt = sys_prompt.replace("[TEKS_ATURAN_BAD]", teks_aturan_bad)
    sys_prompt = sys_prompt.replace("[LARANGAN_TAMBAHAN_DINAMIS]", teks_larangan)
    sys_prompt = sys_prompt.replace("[STRUKTUR_RESPONS_DINAMIS]", teks_struktur)

    # SUNTIK KE USER
    usr_prompt = usr_prompt.replace("[LABEL_EMOSI]", emosi.upper())
    usr_prompt = usr_prompt.replace("[FILE_KONTEKS]", teks_konteks_jadi)
    usr_prompt = usr_prompt.replace("[TEKS_KARAKTER_KOGNITIF]", teks_karakter_kognitif)
    usr_prompt = usr_prompt.replace("[TEKS_KARAKTER_EMOSI]", teks_karakter_emosi)
    usr_prompt = usr_prompt.replace("[TEKS_KARAKTER_OOT]", teks_karakter_oot)
    usr_prompt = usr_prompt.replace("[TEKS_KARAKTER_BAD]", teks_karakter_bad)
    usr_prompt = usr_prompt.replace("[PREFIX_SISWA]", prefix_siswa)

    # LOGIKA JUMLAH TURN
    if kognitif == "low" or emosi in ["frustrasi", "bingung"]:
        min_t, max_t = 10, 15
    elif oot != "0x" or bad != "0x":
        min_t, max_t = 7, 10
    else:
        min_t, max_t = 3, 7

    jumlah_turn = str(random.randint(min_t, max_t))
    usr_prompt = usr_prompt.replace("[JUMLAH_TURN]", jumlah_turn)

    return sys_prompt, usr_prompt, int(jumlah_turn)

# CELL 5 — Proses Generate

In [ ]:
import os
import json
import random
import itertools
import time
import requests

# ==========================================
# 1. KONFIGURASI BUKU ABSEN MASTER
# ==========================================
FILE_BUKU_ABSEN = "/content/drive/MyDrive/output_sft_baru/buku_absen_essay_ips.json"

# ==========================================
# 2. TARGET RUN
# ==========================================
TARGET_JUMLAH_DATA = 400
MAX_RETRIES        = 3
jumlah_sukses      = 0

# ==========================================
# TAHAP A: BIKIN BUKU ABSEN (1.667 DATA MATERI)
# ==========================================
if not os.path.exists(FILE_BUKU_ABSEN):
    print("Buku Absen belum ada! Membuat 1.667 tiket kombinasi baru...")

    LIST_KURIKULUM = ["Kurikulum Merdeka", "KTSP", "K-13"]
    LIST_TIPE      = ["essay"]
    OPSI_KOGNITIF  = ["low", "middle", "high"]
    OPSI_EMOSI     = ["antusias", "bingung", "bosan", "frustrasi", "tidak_terdeteksi"]
    OPSI_INSIDEN   = ["0x", "1x", "2x"]

    # 1. Bikin 405 tiket dasar
    kombinasi_dasar = list(itertools.product(
        LIST_KURIKULUM, LIST_TIPE, OPSI_KOGNITIF, OPSI_EMOSI, OPSI_INSIDEN, OPSI_INSIDEN
    ))

    # 2. Pengali untuk mencapai target 1667
    semua_tiket = kombinasi_dasar * 4 # 405 x 4 = 1620
    tiket_bonus = random.choices(kombinasi_dasar, k=47) # sisa 47 tiket
    total_tiket_1667 = semua_tiket + tiket_bonus

    # 3. Bikin tumpukan stempel Gaya Bahasa (1000 Informal, 667 Formal)
    tumpukan_stempel = ["informal"] * 1000 + ["formal"] * 667
    random.shuffle(tumpukan_stempel)

    # 4. Cap stempel ke setiap tiket
    tiket_final_1667 = []
    for i in range(1667):
        tiket_baru = list(total_tiket_1667[i]) + [tumpukan_stempel[i]]
        tiket_final_1667.append(tiket_baru)

    random.shuffle(tiket_final_1667)

    with open(FILE_BUKU_ABSEN, "w") as f:
        json.dump(tiket_final_1667, f)
    print("1.667 Tiket Khusus Materi berhasil dicetak!\n")

# ==========================================
# TAHAP B: AMBIL TIKET DARI BUKU ABSEN
# ==========================================
with open(FILE_BUKU_ABSEN, "r") as f:
    daftar_antrean = json.load(f)

print(f"Sisa tiket keseluruhan di Drive: {len(daftar_antrean)}")

if len(daftar_antrean) == 0:
    print("SELAMAT! Target 1.666 data materimu sudah selesai semua.")
else:
    tiket_diambil = min(TARGET_JUMLAH_DATA, len(daftar_antrean))
    tiket_hari_ini = daftar_antrean[:tiket_diambil]

    daftar_antrean = daftar_antrean[tiket_diambil:]
    with open(FILE_BUKU_ABSEN, "w") as f:
        json.dump(daftar_antrean, f)

    print(f"Memulai generate {len(tiket_hari_ini)} data baru... (Sisa tiket untuk besok: {len(daftar_antrean)})\n")

    tiket_gagal = []

    # ==========================================
    # TAHAP C: PROSES GENERATE KE API LLM
    # ==========================================
    for i, tiket in enumerate(tiket_hari_ini):
        urutan_kurikulum, urutan_tipe, sk_kognitif, sk_emosi, sk_oot, sk_bad, sk_gaya_bahasa = tiket

        PATH_BASE_SYS = os.path.join(ROOT_DIR, f"e_base_system_prompt_{sk_gaya_bahasa}.md")
        PATH_BASE_USR = os.path.join(ROOT_DIR, f"e_base_user_prompt_{sk_gaya_bahasa}.md")

        DIR_KOMPONEN_SYS = os.path.join(ROOT_DIR, f"komponen_system_{sk_gaya_bahasa}/")
        DIR_KOMPONEN_USR = os.path.join(ROOT_DIR, f"komponen_user_{sk_gaya_bahasa}/")

        status_berhasil_per_tiket = False
        current_temp = round(random.uniform(0.75, 0.95), 2)

        # Mencari file JSON materi yang sesuai kurikulum
        kandidat_file = []
        for f in daftar_file_sumber:
            f_norm = f.replace("\\", "/")
            if f"/{urutan_kurikulum}/" in f_norm and f"/{urutan_tipe}/" in f_norm:
                kandidat_file.append(f)

        if len(kandidat_file) > 0:
            file_terpilih = random.choice(kandidat_file)
        else:
            file_terpilih = random.choice(daftar_file_sumber)

        nama_file_saja = os.path.basename(file_terpilih)

        with open(file_terpilih, "r", encoding="utf-8") as f:
            data_json = json.load(f)

        meta = data_json.get("metadata", {})
        task_type = meta.get("task_type", "essay") # <-- UBAH FALLBACK JADI "essay"

        # EKSTRAKSI KHUSUS ESSAY
        teks_konteks_jadi = ""
        isi_assistant = data_json.get("assistant", [])

        if task_type == "essay": # <-- UBAH JADI "essay"
            for idx, soal in enumerate(isi_assistant):
                teks_konteks_jadi += f"\nSoal {idx+1}:\nStimulus: {soal.get('stimulus')}\n"
                teks_konteks_jadi += f"Pertanyaan: {soal.get('question')}\n"
                teks_konteks_jadi += f"Penjelasan: {soal.get('explanation')}\n"

        # Pemanggilan string_evaluasi dihapus total, langsung suntik ke prefix
        prefix_siswa = f'[Emosi: {sk_emosi}]\n[Konteks: "{nama_file_saja}"]'

        print(f"[{i+1}/{len(tiket_hari_ini)}] Kurikulum: {urutan_kurikulum} | Tipe: {task_type.upper()} | Emosi: {sk_emosi} | Kognitif: {sk_kognitif} | OOT:{sk_oot} | Bad:{sk_bad} | Gaya: {sk_gaya_bahasa.upper()}")

        sys_p, usr_p, turn_terpakai = get_prompts_dinamis(sk_kognitif, sk_emosi, sk_oot, sk_bad, teks_konteks_jadi, prefix_siswa)

        payload = {
            "model": MODEL_NAME,
            "messages": [
                {"role": "system", "content": sys_p},
                {"role": "user", "content": usr_p}
            ],
            "temperature": current_temp,
            "max_tokens": 8192
        }
        headers = {"Authorization": f"Bearer {API_KEY}", "Content-Type": "application/json"}

        for attempt in range(MAX_RETRIES):
            try:
                response = requests.post(URL, headers=headers, json=payload, timeout=120)
                if response.status_code == 200:
                    raw_content = response.json().get('choices', [{}])[0].get('message', {}).get('content')

                    raw_content = raw_content.strip()
                    if raw_content.startswith('```json'):
                        raw_content = raw_content[7:]
                    elif raw_content.startswith('```'):
                        raw_content = raw_content[3:]
                    if raw_content.endswith('```'):
                        raw_content = raw_content[:-3]

                    raw_content = raw_content.strip()

                    try:
                        dialog_json = json.loads(raw_content, strict=False)
                    except json.JSONDecodeError:
                        continue

                    pesan_dialog = dialog_json if isinstance(dialog_json, list) else dialog_json.get("messages", [])

                    if pesan_dialog and pesan_dialog[-1]["role"] == "assistant":
                        pesan_percakapan = [msg for msg in pesan_dialog if msg["role"] != "system"]
                        turn_asli = len(pesan_percakapan) // 2

                        hasil_dataset.append({
                            "metadata": {
                                "mapel": meta.get("mata_pelajaran", "IPS"),
                                "kurikulum": meta.get("kurikulum", "Kurikulum Merdeka"),
                                "bab": meta.get("bab_judul", ""),
                                "subbab": meta.get("sub_bab", ""),
                                "tipe_konteks": task_type,
                                "level_konteks": meta.get("level", ""),
                                "skenario_emosi": sk_emosi,
                                "skenario_kognitif": sk_kognitif,
                                "skenario_oot": sk_oot,
                                "skenario_bad": sk_bad,
                                "skenario_gaya_bahasa": sk_gaya_bahasa,
                                "path_file_sumber": file_terpilih,
                                "temp_used": current_temp,
                                "turn": turn_asli,
                                "model": MODEL_NAME
                            },
                            "messages": [
                                {"role": "system", "content": f"ips_sys_{task_type}_{sk_emosi}_{sk_kognitif}_oot{sk_oot}_bad{sk_bad}_{sk_gaya_bahasa}"}
                            ] + pesan_percakapan
                        })
                        jumlah_sukses += 1
                        status_berhasil_per_tiket = True
                        print(f"  -> Berhasil.")
                        time.sleep(3)
                        break
                    else:
                        print("  -> Dialog gagal/berhenti di user. (Mencoba ulang...)")
                else:
                    print(f"  -> Error API {response.status_code}")
                    time.sleep(15)
            except Exception as e:
                print(f"  -> Exception: {e}")
                time.sleep(5)

        if not status_berhasil_per_tiket:
            tiket_gagal.append(tiket)
            print("  -> TIKET GAGAL DIPROSES. Akan dikembalikan ke antrean Drive.\n")

    # ==========================================
    # TAHAP D: REFUND TIKET GAGAL KE DRIVE
    # ==========================================
    if len(tiket_gagal) > 0:
        print(f"\nMengembalikan {len(tiket_gagal)} tiket yang hangus ke Buku Absen di Drive...")

        with open(FILE_BUKU_ABSEN, "r") as f:
            antrean_sekarang = json.load(f)

        antrean_sekarang.extend(tiket_gagal)
        random.shuffle(antrean_sekarang)

        with open(FILE_BUKU_ABSEN, "w") as f:
            json.dump(antrean_sekarang, f)

        print("Refund tiket berhasil!")

    print(f"\nSelesai. Berhasil generate: {jumlah_sukses}/{len(tiket_hari_ini)} percakapan.")

Sisa tiket keseluruhan di Drive: 316
Memulai generate 316 data baru... (Sisa tiket untuk besok: 0)

[1/316] Kurikulum: KTSP | Tipe: ESSAY | Emosi: antusias | Kognitif: high | OOT:2x | Bad:2x | Gaya: FORMAL
  -> Berhasil.
[2/316] Kurikulum: Kurikulum Merdeka | Tipe: ESSAY | Emosi: bingung | Kognitif: low | OOT:0x | Bad:2x | Gaya: INFORMAL
  -> Berhasil.
[3/316] Kurikulum: KTSP | Tipe: ESSAY | Emosi: antusias | Kognitif: middle | OOT:0x | Bad:0x | Gaya: INFORMAL
  -> Berhasil.
[4/316] Kurikulum: K-13 | Tipe: ESSAY | Emosi: antusias | Kognitif: high | OOT:0x | Bad:0x | Gaya: FORMAL
  -> Berhasil.
[5/316] Kurikulum: K-13 | Tipe: ESSAY | Emosi: frustrasi | Kognitif: low | OOT:1x | Bad:0x | Gaya: INFORMAL
  -> Berhasil.
[6/316] Kurikulum: K-13 | Tipe: ESSAY | Emosi: bingung | Kognitif: low | OOT:0x | Bad:2x | Gaya: FORMAL
  -> Berhasil.
[7/316] Kurikulum: KTSP | Tipe: ESSAY | Emosi: tidak_terdeteksi | Kognitif: low | OOT:0x | Bad:1x | Gaya: FORMAL
  -> Berhasil.
[8/316] Kurikulum: Kurikulum 

# CELL 6 — Simpan ke Google Drive

In [ ]:
CUSTOM_OUTPUT_PATH = "/content/drive/MyDrive/output_sft_baru/" # sesuaikan nama folder

if 'hasil_dataset' in globals() and len(hasil_dataset) > 0:
    os.makedirs(CUSTOM_OUTPUT_PATH, exist_ok=True) # Otomatis buat folder jika tidak ada

    # Nama file global
    nama_file  = f"final_essay_{MATA_PELAJARAN}.json"
    full_path  = os.path.join(CUSTOM_OUTPUT_PATH, nama_file)
    data_final = []

    if os.path.exists(full_path):
        try:
            with open(full_path, "r", encoding="utf-8") as f:
                data_final = json.load(f)
        except Exception:
            pass

    # menambahkan data baru ke file yang sudah ada
    data_final.extend(hasil_dataset)

    try:
        with open(full_path, "w", encoding="utf-8") as f:
            json.dump(data_final, f, ensure_ascii=False, indent=4)
        print(f"Tersimpan: {len(data_final)} baris -> {full_path}")
        hasil_dataset.clear()
    except Exception as e:
        print(f"Gagal menyimpan: {e}")

Tersimpan: 1667 baris -> /content/drive/MyDrive/output_sft_baru/final_essay_IPS.json
